# Notebook 06 — Simulation Realism (v2)

This notebook dissects each of the **5 realism improvements** added to `simulate.py` and shows their effect on the generated data.

| # | Improvement | New column(s) |
|---|-------------|---------------|
| 1 | Correlated shadow fading (Gudmundson AR-1) | — affects RSRP smoothness |
| 2 | A3 / A4 / A5 event classification | `event_type` |
| 3 | Cell load + SINR degradation | `serving_cell_load` |
| 4 | Handover failure + RLF | `handover_failure`, `rlf_flag` |
| 5 | Ping-pong detection | `ping_pong` |

In [ ]:
import sys, math
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('../data/raw/dataset.csv')
ho = df[df['handover_event'] == 1].copy()

print(f'Dataset: {len(df):,} rows · {df["ue_id"].nunique()} UEs · {df["timestamp"].nunique()} steps')
print(f'Columns: {df.columns.tolist()}')

---
## Improvement 1 — Correlated Shadow Fading (Gudmundson AR-1)

### What changed
**v1:** independent Gaussian noise added to RSRP at every step → unrealistic rapid jumps  
**v2:** shadow fading evolves as a first-order autoregressive process tied to distance moved

$$\sigma_t = \rho \cdot \sigma_{t-1} + \sqrt{1-\rho^2} \cdot \mathcal{N}(0,\,\sigma^2), \quad \rho = e^{-d/d_{\text{corr}}}$$

- **Pedestrian (1 m/s):** $d=1$ m/step $\Rightarrow$ $\rho = e^{-1/100} \approx 0.990$ — almost frozen  
- **Vehicle (15 m/s):** $d=15$ m/step $\Rightarrow$ $\rho = e^{-15/100} \approx 0.861$ — decorrelates faster

In [ ]:
# Simulate 300 steps of correlated vs iid shadow fading for two UE speeds
rng = np.random.default_rng(0)
SIGMA, DECORR = 2.0, 100.0
N = 300

def simulate_shadow(speed, n=N, seed=0):
    rng = np.random.default_rng(seed)
    shadow = np.zeros(n)
    shadow[0] = rng.normal(0, SIGMA)
    for i in range(1, n):
        d = speed * 1.0
        rho = math.exp(-d / DECORR)
        shadow[i] = rho * shadow[i-1] + math.sqrt(max(1-rho**2, 0)) * rng.normal(0, SIGMA)
    return shadow

iid_noise   = rng.normal(0, SIGMA, N)
ped_shadow  = simulate_shadow(speed=1.0)
veh_shadow  = simulate_shadow(speed=15.0)

fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
for ax, signal, label, color in [
    (axes[0], iid_noise,  'v1 — Independent noise (all speeds)', 'gray'),
    (axes[1], ped_shadow, 'v2 — AR(1) Pedestrian (1 m/s, ρ≈0.990)', 'steelblue'),
    (axes[2], veh_shadow, 'v2 — AR(1) Vehicle (15 m/s, ρ≈0.861)',  'darkorange'),
]:
    ax.plot(signal, color=color, lw=1.5)
    ax.axhline(0, ls='--', color='black', lw=0.8, alpha=0.4)
    ax.set_ylabel('Shadow (dB)'); ax.set_title(label)
    ax.set_ylim(-8, 8)

axes[-1].set_xlabel('Step (s)')
plt.suptitle('Shadow Fading: Independent (v1) vs Gudmundson AR(1) (v2)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Autocorrelation function — correlated fading retains memory
from statsmodels.tsa.stattools import acf

lags = np.arange(0, 40)
acf_iid = acf(iid_noise,  nlags=39, fft=True)
acf_ped = acf(ped_shadow, nlags=39, fft=True)
acf_veh = acf(veh_shadow, nlags=39, fft=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lags, acf_iid, 'o-', color='gray',       lw=1.5, ms=4, label='v1 — iid noise')
ax.plot(lags, acf_ped, 's-', color='steelblue',  lw=1.5, ms=4, label='v2 — pedestrian (ρ≈0.990)')
ax.plot(lags, acf_veh, '^-', color='darkorange', lw=1.5, ms=4, label='v2 — vehicle (ρ≈0.861)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Lag (steps)'); ax.set_ylabel('Autocorrelation')
ax.set_title('ACF of Shadow Fading — AR(1) maintains spatial memory')
ax.legend(); ax.set_ylim(-0.3, 1.05)
plt.tight_layout(); plt.show()

In [ ]:
# Effect on RSRP traces in the actual dataset
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
COLORS = plt.cm.tab10.colors

for i, (ue_id, title, color) in enumerate([
    (0,  'Pedestrian UE 0 (≈1.8 m/s)',  'steelblue'),
    (10, 'Vehicle UE 10 (≈13 m/s)',     'darkorange'),
]):
    ue = df[df['ue_id'] == ue_id].sort_values('timestamp').head(400)
    axes[i].plot(ue['timestamp'], ue['rsrp_serving'], lw=1.5, color=color)
    ho_u = ue[ue['handover_event'] == 1]
    axes[i].scatter(ho_u['timestamp'], ho_u['rsrp_serving'],
                    color='red', s=60, zorder=5, label='HO event')
    axes[i].set_xlabel('Time (s)'); axes[i].set_ylabel('RSRP Serving (dBm)')
    axes[i].set_title(title)
    axes[i].legend(fontsize=9)

plt.suptitle('RSRP Serving — smooth correlated fading visible in v2', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

---
## Improvement 2 — A3 / A4 / A5 Event Classification

### What changed
**v1:** all handovers treated as generic A3 events  
**v2:** each handover classified by the 3GPP measurement event that justified it

| Event | Condition | Meaning |
|-------|-----------|----------|
| **A3** | `RSRP_nb > RSRP_srv + 3 dB` | Standard inter-cell HO — best-server reselection |
| **A4** | `RSRP_nb > −55 dBm` | Neighbor very strong — UE moving close to target BS |
| **A5** | `RSRP_srv < −68` AND `RSRP_nb > −60` | Coverage emergency — serving cell critically weak |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
event_colors = {'A3': '#4e79a7', 'A4': '#f28e2b', 'A5': '#e15759'}

# Pie chart
counts = ho['event_type'].value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[event_colors[e] for e in counts.index],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('HO Event Type Distribution', fontsize=12)

# RSRP serving at trigger, by event type
for ev, color in event_colors.items():
    subset = ho[ho['event_type'] == ev]['rsrp_serving']
    if len(subset) > 0:
        axes[1].hist(subset, bins=20, alpha=0.7, label=ev,
                     color=color, edgecolor='white')
axes[1].axvline(-68, ls='--', color='black', lw=1.2, label='A5 serving threshold')
axes[1].set_xlabel('RSRP Serving at HO (dBm)')
axes[1].set_ylabel('Count')
axes[1].set_title('Serving RSRP at HO — by Event Type')
axes[1].legend(fontsize=9)

# RSRP neighbor at trigger, by event type
for ev, color in event_colors.items():
    subset = ho[ho['event_type'] == ev]['rsrp_neighbor']
    if len(subset) > 0:
        axes[2].hist(subset, bins=20, alpha=0.7, label=ev,
                     color=color, edgecolor='white')
axes[2].axvline(-55, ls='--', color='darkorange', lw=1.2, label='A4 neighbor threshold')
axes[2].axvline(-60, ls=':', color='red', lw=1.2, label='A5 neighbor threshold')
axes[2].set_xlabel('RSRP Neighbor at HO (dBm)')
axes[2].set_title('Neighbor RSRP at HO — by Event Type')
axes[2].legend(fontsize=9)

plt.suptitle('A3 / A4 / A5 Event Type Analysis', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Event type by UE type (vehicle vs pedestrian)
speed_map = df[['ue_id','ue_speed']].drop_duplicates('ue_id').set_index('ue_id')['ue_speed']
ho['ue_type'] = ho['ue_id'].map(lambda u: 'Vehicle' if speed_map[u] > 3 else 'Pedestrian')

ct = pd.crosstab(ho['ue_type'], ho['event_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(7, 4))
ct[[c for c in ['A3','A4','A5'] if c in ct.columns]].plot(
    kind='bar', ax=ax, color=[event_colors[c] for c in ct.columns],
    edgecolor='white', width=0.5
)
ax.set_xlabel(''); ax.set_xticklabels(ct.index, rotation=0, fontsize=11)
ax.set_ylabel('% of HO events'); ax.set_ylim(0, 105)
ax.set_title('Event Type Distribution — Pedestrian vs Vehicle UEs')
ax.legend(title='Event'); plt.tight_layout(); plt.show()

---
## Improvement 3 — Cell Load & SINR Degradation

### What changed
**v1:** SINR depends only on inter-cell interference from other BSs  
**v2:** each co-channel UE on the same cell adds **1.5 dB** SINR penalty

$$SINR_{\text{v2}} = SINR_{\text{baseline}} - 1.5 \times (N_{\text{co-cell UEs}} - 1)$$

This makes SINR sensitive to instantaneous cell load — a realistic LTE behaviour.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Cell load distribution
load_counts = df['serving_cell_load'].value_counts().sort_index()
axes[0].bar(load_counts.index, load_counts.values,
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('UEs on serving cell'); axes[0].set_ylabel('Row count')
axes[0].set_title('Serving Cell Load Distribution')
axes[0].axvline(df['serving_cell_load'].mean(), ls='--', color='red',
                lw=1.5, label=f'Mean = {df["serving_cell_load"].mean():.1f}')
axes[0].legend()

# SINR vs cell load
sinr_by_load = df.groupby('serving_cell_load')['sinr'].agg(['mean','std'])
axes[1].errorbar(sinr_by_load.index, sinr_by_load['mean'],
                 yerr=sinr_by_load['std'], fmt='o-',
                 color='steelblue', capsize=4, lw=2)
# Theoretical degradation line
loads = np.arange(1, sinr_by_load.index.max() + 1)
baseline = sinr_by_load['mean'].iloc[0]
axes[1].plot(loads, baseline - 1.5 * (loads - 1), 'r--',
             lw=1.5, label='−1.5 dB/UE model')
axes[1].set_xlabel('Serving cell load (UEs)')
axes[1].set_ylabel('Mean SINR (dB)')
axes[1].set_title('SINR Degradation with Cell Load')
axes[1].legend(fontsize=9)

# CQI vs cell load
cqi_by_load = df.groupby('serving_cell_load')['cqi'].mean()
axes[2].bar(cqi_by_load.index, cqi_by_load.values,
            color='seagreen', edgecolor='white')
axes[2].set_xlabel('Serving cell load (UEs)')
axes[2].set_ylabel('Mean CQI')
axes[2].set_title('CQI Degradation with Cell Load')

plt.suptitle('Cell Load Impact on Radio Quality', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Cell load over time per cell
load_pivot = df.groupby(['timestamp','serving_cell_id']).size().unstack(fill_value=0)
load_sampled = load_pivot.iloc[::10]  # every 10 steps

fig, ax = plt.subplots(figsize=(13, 3))
for cell_id in sorted(load_pivot.columns):
    ax.plot(load_sampled.index, load_sampled[cell_id],
            lw=1.5, label=f'Cell {cell_id}')
ax.set_xlabel('Time (s)'); ax.set_ylabel('UEs on cell')
ax.set_title('Cell Load Over Time — UEs migrate between cells as handovers occur')
ax.legend(ncol=4, fontsize=9); plt.tight_layout(); plt.show()

---
## Improvement 4 — Handover Failure & Radio Link Failure (RLF)

### What changed
**v1:** every handover attempt always succeeds  
**v2:** handover success probability follows a sigmoid of serving RSRP

$$P(\text{fail}) = \sigma\!\left(\frac{-(RSRP_{\text{srv}} - (-80))}{5}\right)$$

| Serving RSRP | P(HO fail) |
|---|---|
| −60 dBm | 1.8% |
| −70 dBm | 12% |
| −80 dBm | 50% |
| −90 dBm | 88% |

After a failed HO, the UE enters **RLF recovery** for 3 steps during which SINR is floored at −15 dB.

In [ ]:
rsrp_range = np.linspace(-100, -50, 200)
p_fail = 1 / (1 + np.exp((rsrp_range - (-80)) / 5))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Sigmoid curve
axes[0].plot(rsrp_range, p_fail * 100, color='tomato', lw=2.5)
for rsrp, label in [(-60,'−60'), (-70,'−70'), (-80,'−80'), (-90,'−90')]:
    pf = 100 / (1 + np.exp((rsrp - (-80)) / 5))
    axes[0].scatter([rsrp], [pf], zorder=5, s=60, color='black')
    axes[0].annotate(f'{pf:.0f}%', (rsrp, pf), textcoords='offset points',
                     xytext=(6, 4), fontsize=9)
axes[0].set_xlabel('Serving RSRP at HO (dBm)')
axes[0].set_ylabel('P(HO failure) %')
axes[0].set_title('Handover Failure Probability')

# RSRP distribution: failed vs successful HOs
ho_succ = ho[ho['handover_failure'] == 0]['rsrp_serving']
ho_fail = ho[ho['handover_failure'] == 1]['rsrp_serving']
axes[1].hist(ho_succ, bins=25, alpha=0.7, label=f'Success (n={len(ho_succ)})',
             color='steelblue', edgecolor='white')
if len(ho_fail) > 0:
    axes[1].hist(ho_fail, bins=10, alpha=0.85, label=f'Failed (n={len(ho_fail)})',
                 color='tomato', edgecolor='white')
axes[1].set_xlabel('Serving RSRP at HO (dBm)')
axes[1].set_ylabel('Count')
axes[1].set_title('RSRP at HO: Successful vs Failed')
axes[1].legend(fontsize=9)

# RLF signature in SINR trace
rlf_ues = df[df['rlf_flag'] == 1]['ue_id'].unique()
if len(rlf_ues) > 0:
    ue_id = rlf_ues[0]
    rlf_t = df[(df['ue_id'] == ue_id) & (df['rlf_flag'] == 1)]['timestamp'].iloc[0]
    trace = df[(df['ue_id'] == ue_id) &
               df['timestamp'].between(rlf_t - 8, rlf_t + 10)]
    axes[2].plot(trace['timestamp'], trace['sinr'], lw=2, color='steelblue')
    rlf_rows = trace[trace['rlf_flag'] == 1]
    axes[2].axhspan(-16, -14, alpha=0.25, color='tomato', label='RLF recovery (SINR = −15 dB)')
    axes[2].scatter(rlf_rows['timestamp'], rlf_rows['sinr'],
                    color='tomato', s=60, zorder=5)
    axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('SINR (dB)')
    axes[2].set_title(f'RLF Signature in SINR — UE {ue_id}')
    axes[2].legend(fontsize=9)

plt.suptitle('Handover Failure & Radio Link Failure', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## Improvement 5 — Ping-Pong Detection

### What changed
**v1:** no awareness of HO history — every HO is treated independently  
**v2:** each HO is checked against the last 10-step history; if the UE is handing BACK to a cell it recently left, it's flagged as a ping-pong

Ping-pong is a real LTE KPI tracked by network operators — it indicates the handover margin is too tight or the UE is oscillating on a cell boundary.

In [ ]:
pp = ho[ho['ping_pong'] == 1]
print(f'Total HO attempts : {len(ho)}')
print(f'Ping-pong HOs     : {len(pp)}  ({100*len(pp)/len(ho):.1f}%)')

if len(pp) > 0:
    print('\nPing-pong events:')
    print(pp[['timestamp','ue_id','serving_cell_id','target_cell_id',
               'rsrp_serving','rsrp_neighbor','event_type']].to_string(index=False))

    # Show a ping-pong in context
    ue_id = int(pp['ue_id'].iloc[0])
    t_pp  = int(pp['timestamp'].iloc[0])
    window = df[(df['ue_id'] == ue_id) & df['timestamp'].between(t_pp - 15, t_pp + 5)]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(window['timestamp'], window['rsrp_serving'],  lw=2, label='Serving RSRP')
    ax.plot(window['timestamp'], window['rsrp_neighbor'], lw=2, ls='--', label='Neighbor RSRP')

    for _, row in window[window['handover_event'] == 1].iterrows():
        color = 'tomato' if row['ping_pong'] else 'green'
        label = f'{"Ping-pong" if row["ping_pong"] else "Normal"} HO (cell {int(row["serving_cell_id"])}→{int(row["target_cell_id"])})'  
        ax.axvline(row['timestamp'], color=color, ls=':', lw=2, label=label)

    ax.set_xlabel('Time (s)'); ax.set_ylabel('RSRP (dBm)')
    ax.set_title(f'UE {ue_id} — Ping-Pong Handover in RSRP Context')
    ax.legend(fontsize=9); plt.tight_layout(); plt.show()

---
## Summary — New Columns at a Glance

In [ ]:
new_cols = ['serving_cell_load', 'event_type', 'handover_failure', 'ping_pong', 'rlf_flag']

print('New column statistics:')
for col in new_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f'  {col:22s}: {df[col].value_counts().to_dict()}')
    else:
        print(f'  {col:22s}: mean={df[col].mean():.3f}  '
              f'min={df[col].min():.0f}  max={df[col].max():.0f}  '
              f'sum={df[col].sum():.0f}')

print(f'\nCorrelation of new numeric columns with handover_soon:')
numeric_new = ['serving_cell_load', 'handover_failure', 'ping_pong', 'rlf_flag']
for col in numeric_new:
    corr = df[col].corr(df['handover_soon'])
    print(f'  {col:22s}: r = {corr:+.4f}')

In [ ]:
# Side-by-side comparison: new columns by handover_soon label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cell load distribution by label
for label, color, name in [(0,'steelblue','No HO soon'), (1,'tomato','HO soon')]:
    subset = df[df['handover_soon'] == label]['serving_cell_load'].value_counts().sort_index()
    axes[0].plot(subset.index, subset.values / subset.sum(),
                 'o-', color=color, lw=2, ms=6, label=name)
axes[0].set_xlabel('Serving cell load'); axes[0].set_ylabel('Proportion')
axes[0].set_title('Cell Load Distribution by Label'); axes[0].legend()

# Event type by label (of HO rows)
event_label = pd.crosstab(
    ho['event_type'],
    ho['handover_failure'].map({0:'Success', 1:'Failure'}),
    normalize='index'
) * 100
event_label.plot(kind='bar', ax=axes[1], color=['steelblue','tomato'],
                 edgecolor='white', width=0.5)
axes[1].set_xlabel('Event type'); axes[1].set_ylabel('% of HO events')
axes[1].set_title('HO Failure Rate by Event Type')
axes[1].set_xticklabels(event_label.index, rotation=0)
axes[1].legend(title='Outcome')

plt.tight_layout(); plt.show()

## Key Takeaways

| Improvement | What it adds to the dataset |
|---|---|
| Correlated shadow fading | Smoother, physically realistic RSRP traces — no step-jumps |
| A3/A4/A5 event types | Richer handover context — severity and cause of each HO |
| Cell load | SINR reflects congestion — more realistic degradation |
| HO failure + RLF | Models real failure modes — RLF signature visible in SINR |
| Ping-pong | Network quality KPI — identifies poor margin configurations |

→ **Next:** [07_model_impact.ipynb](07_model_impact.ipynb)